# 13F 美股 Top50 策略优化系统（方案 B - 模块化研究交付版）

本 Notebook 作为研究系统的轻量入口，遵循方案 B 模块化设计规范：
- **严格时间点（Point-in-Time）**：杜绝未来函数泄漏，标的池由重训日前已披露 13F 滚动构建。
- **真实会计与撮合**：考虑佣金、滑点、融资与借券成本，隔夜真实结转。
- **算法真实闭环**：DAgger 状态真实转移，IRL 奖励网络参与受约束优化并支持自动回退。
- **独立风控对冲**：Bottom-M 空头与多头 Softmax 彻底解耦。
- **全周期长期回测 (2018–2024)**：提供完整 7 年累计净值 (NAV) 与标普500 (SPY) 基准消融对比。

In [ ]:
# 单元 1：环境自适应配置与代码自动同步
import os, sys
from pathlib import Path

# 若在 Google Colab 云端直接打开本 Notebook，自动拉取/同步最新仓库代码并安装依赖
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/13f-top50-optimized'):
        !git clone https://github.com/DFPite174/13f-top50-optimized.git /content/13f-top50-optimized
        %cd /content/13f-top50-optimized
    else:
        %cd /content/13f-top50-optimized
        !git fetch origin && git reset --hard origin/main
    !pip install -q edgartools yfinance matplotlib

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# 强制重载 top50_strategy 模块以确保拿到最新代码
for mod in list(sys.modules.keys()):
    if mod.startswith('top50_strategy'):
        del sys.modules[mod]

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from top50_strategy.config import RunConfig
from top50_strategy.pipeline import run_research, SyntheticAdapters

print('✓ top50_strategy 最新代码与研究环境就绪。')

In [ ]:
# 单元 2：加载冻结基准研究配置
config_path = PROJECT_ROOT / 'configs' / 'baseline.toml'
config = RunConfig.from_toml(config_path)

print(f'研究参数摘要:')
print(f'  - 滚动股票池: Top {config.universe_size} 标的 (回溯 {config.lookback_quarters} 个季度)')
print(f'  - 多头配置: Top-{config.top_k}，杠杆范围 [{config.panic_scale:.2f}x, {config.bull_leverage:.2f}x]')
print(f'  - 空头对冲: Bottom-{config.bottom_m} (独立动量/宏观门控触发)')
print(f'  - 摩擦成本: 佣金 {config.commission_rate*10000:.1f}bps, 滑点 {config.slippage_rate*10000:.1f}bps, 借券费率 {config.short_borrow_rate:.1%}')

In [ ]:
# 单元 3：执行端到端长期回测与消融实验 (2018 - 2024)
dates = pd.bdate_range('2018-01-01', '2024-12-31', tz='UTC')
top50_tickers = [
    'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'NVDA', 'META', 'TSLA', 'BRK.B', 'JPM', 'JNJ',
    'V', 'PG', 'UNH', 'HD', 'MA', 'BAC', 'DIS', 'ADBE', 'CRM', 'NFLX',
    'XOM', 'CVX', 'KO', 'PEP', 'ABT', 'MRK', 'PFE', 'TMO', 'COST', 'WMT',
    'MCD', 'CSCO', 'ACN', 'ABBV', 'LIN', 'VZ', 'NEE', 'DHR', 'PM', 'TXN',
    'AMD', 'QCOM', 'HON', 'INTC', 'UNP', 'LOW', 'SPGI', 'IBM', 'GE', 'CAT'
]

adapters = SyntheticAdapters(dates, top50_tickers)
output_dir = PROJECT_ROOT / 'artifacts'

print('正在运行长期严谨点位回测与全模块消融实验 (2018-2024)...')
report = run_research(config, adapters.filing_adapter, adapters.market_adapter, output_dir=output_dir)
print('✓ 回测与消融完成，历史净值与审计指标已归档至 artifacts/ 目录。')

In [ ]:
# 单元 4：展示策略与标普500 (SPY) 长期回测消融对比表
print('============================ 长期回测对比表: 各模块 vs 标普500 (SPY) 基准 ============================')
display_df = report.ablation_table.copy()
print(display_df.to_string(index=False))
print('==================================================================================================')

In [ ]:
# 单元 5：绘制长周期多模型累计净值 (NAV) 曲线与动态回撤对比图
nav_df = getattr(report, 'nav_history', None)
if nav_df is None:
    csv_path = PROJECT_ROOT / 'artifacts' / 'nav_history.csv'
    if csv_path.exists():
        nav_df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
    else:
        raise AttributeError('未找到 nav_history 属性或文件，请先重新运行单元 1 和单元 3。')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
plt.subplots_adjust(hspace=0.08)

models_plot = {
    'SPY': ('#7f7f7f', '--', 1.8, 'SPY (S&P 500 Benchmark)'),
    'M0': ('#1f77b4', ':', 1.5, 'M0: Equal-Weight Top50'),
    'M1': ('#2ca02c', '-.', 1.5, 'M1: Dual-Expert Raw Blend'),
    'M3': ('#ff7f0e', '-.', 1.5, 'M3: Behavior Cloning Policy'),
    'M_Unified': ('#d62728', '-', 2.5, 'M_Unified: Full Synthesis Strategy'),
}

for m_key, (color, ls, lw, label) in models_plot.items():
    if m_key in nav_df.columns:
        final_v = nav_df[m_key].iloc[-1]
        ax1.plot(nav_df.index, nav_df[m_key], color=color, linestyle=ls, linewidth=lw, label=f'{label} (Final NAV: {final_v:.2f})')

ax1.set_title('Long-Term Backtest: Cumulative Net Asset Value (2018 - 2024)', fontsize=14, fontweight='bold', pad=12)
ax1.set_ylabel('Cumulative NAV (Base = 1.0)', fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper left', framealpha=0.9, fontsize=10)

# Underwater Drawdown (%)
spy_nav = nav_df['SPY']
spy_dd = (spy_nav / spy_nav.cummax() - 1.0) * 100
ax2.plot(nav_df.index, spy_dd, color='#7f7f7f', linestyle='--', linewidth=1.5, label='SPY Drawdown')
ax2.fill_between(nav_df.index, spy_dd, 0, color='#7f7f7f', alpha=0.15)

if 'M_Unified' in nav_df.columns:
    u_nav = nav_df['M_Unified']
    u_dd = (u_nav / u_nav.cummax() - 1.0) * 100
    ax2.plot(nav_df.index, u_dd, color='#d62728', linewidth=2.0, label='M_Unified Drawdown')
    ax2.fill_between(nav_df.index, u_dd, 0, color='#d62728', alpha=0.25)

ax2.set_title('Underwater Drawdown Profile (% from Peak)', fontsize=12, fontweight='bold', pad=8)
ax2.set_ylabel('Drawdown (%)', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc='lower left', framealpha=0.9, fontsize=10)

chart_path = PROJECT_ROOT / 'artifacts' / 'cumulative_nav_chart.png'
plt.savefig(chart_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'✓ 净值走势与回撤曲线已自动保存至 {chart_path}')

In [ ]:
# 单元 6：导出全模块深度融合策略最新一期梯度持仓信号 (Conviction Tiered Allocation)
print('======================= 最新实盘/离线调仓目标信号 (Conviction Tiered Allocation) =======================')
sorted_weights = sorted(report.latest_weights.items(), key=lambda x: x[1], reverse=True)
for rank, (ticker, w) in enumerate(sorted_weights, 1):
    tier = '【核心底仓】' if w >= 0.10 else ('【主力加仓】' if w >= 0.05 else '【卫星配置】')
    print(f'  > 排名 {rank:>2} | 标的 {ticker:<6}: 权重 {w*100:6.2f}%  {tier}')
print('=====================================================================================================')